<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch18_ethics_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>⚖️ Chapter 18 — Ethics and Responsible AI</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 50 mins | Level: All Levels</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Identify sources of bias in ML pipelines
- Measure fairness metrics across demographic groups
- Apply SHAP values to explain individual predictions
- Understand GDPR/data privacy implications for ML systems
- Design checklists for responsible AI deployment

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble        import RandomForestClassifier
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import accuracy_score

%matplotlib inline
np.random.seed(42)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Synthetic Indian Loan Dataset with Demographic Features
# ─────────────────────────────────────────────────────────────
n = 1200
np.random.seed(42)

gender   = np.random.choice(['Male', 'Female'], n, p=[0.65, 0.35])
age      = np.random.randint(22, 60, n)
income   = np.where(gender=='Male',
                    np.random.lognormal(11.5, 0.5, n),
                    np.random.lognormal(11.2, 0.5, n)).astype(int)  # intentional pay gap
credit   = np.random.normal(700, 70, n).clip(300, 850).astype(int)
education= np.random.choice(['Graduate', 'Not Graduate'], n, p=[0.70, 0.30])

# Default: correlated mainly with credit + income, but income has gender bias baked in
prob_default = 1 / (1 + np.exp(-(-0.008*credit + -0.0000015*income + 2.5)))
default  = (np.random.rand(n) < prob_default).astype(int)

df = pd.DataFrame({
    'gender': gender, 'age': age, 'income': income,
    'credit_score': credit, 'education': education, 'default': default
})

print(f'Dataset: {df.shape}')
print(f'Default rate by gender:')
print(df.groupby('gender')['default'].mean().round(3))

## 📖 Section A — Types of Bias in ML

| Bias Type | Definition | Example |
|-----------|-----------|--------|
| **Historical bias** | Patterns in training data reflect historical discrimination | Hiring model trained on mostly male engineers predicts male is better |
| **Representation bias** | Training data under-represents a group | Face recognition fails on dark skin tones (rare in training data) |
| **Measurement bias** | Proxy features carry demographic information | ZIP code as a proxy for race in credit scoring |
| **Feedback loop** | Model decisions create data that reinforces model | Loan refusals in some areas → fewer successful loans there → model sees lower success rate → more refusals |

> 💡 **Key Rule:** Removing sensitive attributes (gender, caste, religion) does NOT prevent bias if other features correlate with them. Income, address, and employment history can all be proxies for demographic groups.

---
## 🟢 Exercise 18.1 — Fairness Audit *(Level 1 · Est. 10 min)*

> Train a loan default classifier WITHOUT gender as a feature. Measure approval rate (predicted = 0) and accuracy separately for Male and Female. Is the model fair?

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 18.1: Fairness Audit
# ─────────────────────────────────────────────────────────────

# Encode education
df_enc = df.copy()
df_enc['education'] = (df_enc['education'] == 'Graduate').astype(int)

# Features WITHOUT gender
X_fair = df_enc[['age', 'income', 'credit_score', 'education']]
y_fair = df_enc['default']

X_tr, X_te, y_tr, y_te = train_test_split(X_fair, y_fair, test_size=0.3, random_state=42)
gender_test = df_enc.loc[X_te.index, 'gender']

# YOUR CODE HERE — scale, train Random Forest, predict

# YOUR CODE HERE — measure accuracy and approval rate by gender

print('Gender-wise fairness metrics:')
# YOUR CODE HERE — print table

print('\n✅ Fairness audit complete!')

In [ ]:
# @title ✅ Solution — Exercise 18.1 (click to expand)

df_enc = df.copy()
df_enc['education'] = (df_enc['education'] == 'Graduate').astype(int)

X_fair = df_enc[['age', 'income', 'credit_score', 'education']]
y_fair = df_enc['default']

X_tr, X_te, y_tr, y_te = train_test_split(X_fair, y_fair, test_size=0.3, random_state=42, stratify=y_fair)
gender_test = df_enc.loc[X_te.index, 'gender']

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_tr_s, y_tr)
y_pred = rf.predict(X_te_s)

fairness = []
for g in ['Male', 'Female']:
    mask = (gender_test == g)
    acc = accuracy_score(y_te[mask], y_pred[mask])
    approval_rate = (y_pred[mask] == 0).mean()   # predicted no-default = approved
    actual_default_rate = y_te[mask].mean()
    fairness.append({'Gender': g, 'Accuracy': round(acc, 3),
                     'Approval Rate': round(approval_rate, 3),
                     'Actual Default Rate': round(actual_default_rate, 3)})

fairness_df = pd.DataFrame(fairness)
print('Gender-wise fairness metrics:')
print(fairness_df.to_string(index=False))
print('\nKey question: Is the gap in approval rates justified by actual default rates?')
print('If Male and Female have similar default rates but very different approval rates → unfair model.')

---
## 🟡 Exercise 18.2 — Feature Importance and Proxy Variables *(Level 2 · Est. 15 min)*

> Plot feature importance. Check if 'income' is a top feature. Simulate 'removing the proxy' by dropping income — does approval rate gap narrow?

In [ ]:
# @title ✅ Solution — Exercise 18.2 (click to expand)

feat_imp = pd.Series(rf.feature_importances_, index=['age', 'income', 'credit_score', 'education'])
feat_imp.sort_values().plot(kind='barh', color='steelblue')
plt.title('Feature Importance (includes potential proxy: income)')
plt.tight_layout()
plt.show()

# Remove income (gender proxy)
X_no_income = df_enc[['age', 'credit_score', 'education']]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_no_income, y_fair, test_size=0.3, random_state=42, stratify=y_fair)
gender_test2 = df_enc.loc[X_te2.index, 'gender']

sc2 = StandardScaler()
rf2 = RandomForestClassifier(n_estimators=100, random_state=42)
rf2.fit(sc2.fit_transform(X_tr2), y_tr2)
y_pred2 = rf2.predict(sc2.transform(X_te2))

fairness2 = []
for g in ['Male', 'Female']:
    mask = (gender_test2 == g)
    fairness2.append({'Gender': g,
                      'Approval Rate (without income)': round((y_pred2[mask]==0).mean(), 3)})

print('Approval rates after removing income proxy:')
print(pd.DataFrame(fairness2).to_string(index=False))
print('\n✅ Removing a proxy variable often reduces — but rarely eliminates — the fairness gap.')
print('True fairness requires examining the entire data collection and labelling process.')

---
## 🔴 Exercise 18.3 — Responsible AI Deployment Checklist *(Level 3 · Est. 15 min)*

> Apply the checklist below to the loan default model. For each item, write 1-2 sentences on status and what is still missing.

### Responsible AI Deployment Checklist

| # | Checkpoint | Status for this model |
|---|----------|---------------------|
| 1 | Data provenance documented | ❓ Synthetic — in production, document source, collection date, consent |
| 2 | Class and demographic balance verified | ❓ Check default rate across gender, age groups, cities |
| 3 | Model performance disaggregated by subgroup | ❓ Done in Ex 18.1 — male vs female accuracy reported |
| 4 | Sensitive attributes and proxies identified | ❓ Income identified as gender proxy in Ex 18.2 |
| 5 | Model explanation method in place | ❓ Use SHAP or feature importance for RBI compliance |
| 6 | Adverse impact analysis done | ❓ Formal statistical test (4/5 rule) for protected group disparate impact |
| 7 | Human review process for edge cases | ❓ Not implemented — should be required for borderline predictions |
| 8 | Data subject rights (DPDP Act 2023 India) | ❓ Customers must be told an AI decision was made and can request review |
| 9 | Monitoring and drift detection in place | ✅ Chapter 17 drift monitoring implemented |
| 10 | Retrain frequency and policy defined | ❓ Define trigger: performance drops >5% or data distribution shifts |

In [ ]:
# ─────────────────────────────────────────────────────────────
# Bonus: 4/5 Rule (Adverse Impact) Statistical Test
# ─────────────────────────────────────────────────────────────
# The 4/5 Rule: if approval rate of disadvantaged group < 80% of
# majority group's approval rate → adverse impact flagged.

male_approval   = (y_pred[gender_test == 'Male']   == 0).mean()
female_approval = (y_pred[gender_test == 'Female'] == 0).mean()

ratio = female_approval / male_approval
print(f'Male approval rate:   {male_approval:.2%}')
print(f'Female approval rate: {female_approval:.2%}')
print(f'Adverse impact ratio: {ratio:.3f}')
print()
if ratio < 0.80:
    print('⚠️ ADVERSE IMPACT DETECTED (ratio < 0.80)')
    print('   Action required: investigate feature contributions, consult legal and HR.')
else:
    print('✅ No adverse impact detected (ratio ≥ 0.80)')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: Can a model be biased even if it doesn't use sensitive attributes?</strong></summary>

**Answer:** Yes — this is proxy discrimination. Features like postal code, income, device type, or browsing history can correlate strongly with race, religion, or gender. A model trained without 'caste' can still produce caste-biased decisions if it uses features that proxy for caste. The only defence is to measure outcomes across demographic groups, not just features. Fairness must be demonstrated empirically — you cannot assume it.
</details>

<details>
<summary><strong>Q2: What does India's Digital Personal Data Protection (DPDP) Act 2023 say about ML?</strong></summary>

**Answer:** The DPDP Act requires: (1) Explicit informed consent for data collection. (2) Purpose limitation — data can only be used for the stated purpose. (3) Data minimisation — collect only what's needed. (4) Right to correction and erasure. (5) Data fiduciary (the company) must ensure data quality and security. For ML systems making significant decisions (loans, hiring), users can request human review. Violations can attract penalties up to ₹250 crore.
</details>

<details>
<summary><strong>Q3: How do you explain an individual ML prediction to a customer?</strong></summary>

**Answer:** Use SHAP (SHapley Additive exPlanations) — it shows how each feature pushed the prediction above or below the baseline. For a rejected loan: 'Your credit score (580) reduced approval probability by 18%. Your income level (₹3.2L) reduced it by 12%. Your 2 existing loans reduced it by 10%.' This is actionable, legally compliant, and understandable. Feature importance (global) tells you overall model behaviour; SHAP tells you WHY this specific person got this specific prediction.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 18 Complete!</h3>
<ul>
<li>Types of ML bias: historical, representation, measurement, feedback loop</li>
<li>Fairness audit: disaggregated accuracy and approval rates</li>
<li>Adverse impact analysis using the 4/5 rule</li>
<li>Responsible AI checklist and India's DPDP Act</li>
</ul>
<p><strong>Next:</strong> Chapter 19 — Time Series Analysis</p>
</div>